---

<h1 align="center">Assignment-6 [Vidur Dua (23124119)]</h1>

---


# Lab Assignment 6: KNN and Logistic Regression

## Objective
The objective of this assignment is to:
- Implement the K-Nearest Neighbors (KNN) algorithm from scratch.
- Optimize the value of K.
- Compare KNN with Logistic Regression using scikit-learn.

## Dataset
We are using a Dengue disease dataset containing symptoms and classification labels.

## Tasks
1. Implement KNN manually.
2. Optimize value of K.
3. Compare KNN with Logistic Regression.

In [20]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [21]:
# Load dataset
df = pd.read_csv("Dengue_diseases_dataset_modified (1).csv")

# Show first 10 rows
df.head(10)

,age,gender,hemoglobin_g_dl,wbc_count,differential_count,rbc_count,platelet_count,platelet_distribution_width,dengue_label
0,43,Male,12.6,2200.0,1,1,62000.0,11.0,1
1,45,Male,13.2,3000.0,0,1,17000.0,17.0,1
2,50,Female,11.0,3300.0,1,1,19000.0,16.3,1
3,57,Female,11.9,3500.0,1,0,29000.0,14.0,1
4,51,Female,13.0,3100.0,0,1,30000.0,14.5,1
5,61,Male,15.0,3300.0,1,1,34000.0,20.0,1
6,6,Child,11.0,2300.0,1,0,69000.0,12.5,1
7,21,Male,14.0,2500.0,1,1,77000.0,13.3,1
8,29,Male,15.0,2400.0,1,1,78000.0,14.5,1
9,31,Female,14.2,3700.0,0,1,82000.0,15.6,1


## Dataset Exploration

- Features: All columns except target
- Target: Disease / Classification column
- Check missing values and data types

In [22]:
# Check missing values
df.isnull().sum()

# Data types
df.dtypes

# Target distribution
df.iloc[:, -1].value_counts()

dengue_label
1    644
0    345
Name: count, dtype: int64

In [23]:
# Convert categorical to numeric if needed
df = pd.get_dummies(df)

# Split features and target
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# Train-test split (70-30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [24]:
from sklearn.impute import SimpleImputer

# Create imputer (mean strategy for numeric data)
imputer = SimpleImputer(strategy='mean')

# Fit on training data and transform both
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [25]:
# Euclidean Distance
def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

In [26]:
# KNN function
def knn_predict(X_train, y_train, X_test, k):
    predictions = []
    
    for test_point in X_test:
        distances = []
        
        for i in range(len(X_train)):
            dist = euclidean_distance(test_point, X_train[i])
            distances.append((dist, y_train[i]))
        
        # Sort distances
        distances.sort(key=lambda x: x[0])
        
        # Get top k neighbors
        neighbors = distances[:k]
        labels = [label for _, label in neighbors]
        
        # Majority vote
        prediction = Counter(labels).most_common(1)[0][0]
        predictions.append(prediction)
    
    return predictions

In [27]:
# Test KNN
k = 5
y_pred = knn_predict(X_train, y_train, X_test, k)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy (KNN):", accuracy)

Accuracy (KNN): 0.5151515151515151


## Choosing Optimal K

Criterion:
- Try different K values
- Choose K with highest accuracy

In [28]:
k_values = range(1, 21)
accuracies = []

for k in k_values:
    y_pred = knn_predict(X_train, y_train, X_test, k)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)

# Best K
best_k = k_values[np.argmax(accuracies)]
print("Best K:", best_k)
print("Best Accuracy:", max(accuracies))

Best K: 3
Best Accuracy: 0.5286195286195287


In [29]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train, y_train)

y_pred_knn = knn_model.predict(X_test)

print("Scikit-learn KNN Accuracy:", accuracy_score(y_test, y_pred_knn))

Scikit-learn KNN Accuracy: 0.5286195286195287


In [30]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))

Logistic Regression Accuracy: 1.0


In [31]:
print("Manual KNN Accuracy:", accuracy)
print("Scikit-learn KNN Accuracy:", accuracy_score(y_test, y_pred_knn))
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))

Manual KNN Accuracy: 0.5151515151515151
Scikit-learn KNN Accuracy: 0.5286195286195287
Logistic Regression Accuracy: 1.0
